# 🤖 Demo 03 — Pipeline RAG Complet FinRAG

Démonstration du pipeline RAG de bout en bout sur des questions financières.

**Objectifs :**
- Pipeline complet ingestion → retrieval → génération
- Questions simples et complexes
- Analyse des citations et de la confiance
- Comparaison avec/sans re-ranking

In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path('.').resolve().parent
sys.path.insert(0, str(ROOT_DIR))

from src.retrieval.vector_store import FinancialVectorStore
from src.retrieval.retriever import HybridFinancialRetriever
from src.retrieval.reranker import CrossEncoderReRanker
from src.generation.generator import FinancialAnswerGenerator
from src.agents.rag_agent import FinancialRAGAgent

# Initialize system
vs = FinancialVectorStore()
retriever = HybridFinancialRetriever(vector_store=vs, top_k=10)
reranker = CrossEncoderReRanker(top_k=5)
generator = FinancialAnswerGenerator()
agent = FinancialRAGAgent(vs, retriever, reranker, generator)

stats = vs.get_stats()
print(f'Système initialisé: {stats["total_chunks"]} chunks')

## 1. Question Simple

In [ ]:
from IPython.display import Markdown, display

question1 = "Quel est le chiffre d'affaires d'Apple en FY2023 ?"

answer1 = agent.answer(question1, use_decomposition=False)

print(f'Question: {question1}')
print(f'Confiance: {answer1.confidence_score:.0%}')
print(f'Temps: {answer1.processing_time:.2f}s')
print(f'Tokens: {answer1.tokens_used:,}')
print(f'Sources: {answer1.context_docs_count}')
print()
display(Markdown(answer1.answer))

## 2. Citations et Sources

In [ ]:
print(f'Citations ({len(answer1.citations)}):')  
for i, cit in enumerate(answer1.citations, 1):
    print(f'  [{i}] {cit.source_file}')
    if cit.page_number:
        print(f'      Page: {cit.page_number}')
    if cit.date:
        print(f'      Date: {cit.date}')
    print(f'      Score: {cit.relevance_score:.3f}')
    print(f'      Extrait: "{cit.excerpt[:80]}..."')
    print()

## 3. Question avec Filtre Temporel

In [ ]:
question2 = "Quelle est la marge opérationnelle de Microsoft ?"

answer2 = agent.answer(
    question2,
    date_range=("2023", "2025"),
    ticker="MSFT",
    use_decomposition=False,
)

print(f'Question: {question2}')
print(f'Filtre: MSFT, 2023-2025')
print()
display(Markdown(answer2.answer))

## 4. Analyse des Métriques de Réponse

In [ ]:
import plotly.graph_objects as go

questions = [
    "Quel est le CA d'Apple en FY2023 ?",
    "Quelle est la croissance d'Azure au T4 FY2024 ?",
    "Quel est l'EPS dilué d'Apple en FY2023 ?",
]

results_data = []
for q in questions:
    a = agent.answer(q, use_decomposition=False)
    results_data.append({
        'question': q[:40] + '...',
        'confidence': a.confidence_score,
        'time': a.processing_time,
        'n_citations': len(a.citations),
    })
    print(f'Q: {q[:50]}')
    print(f'   Confiance: {a.confidence_score:.0%} | Temps: {a.processing_time:.2f}s | Citations: {len(a.citations)}')

print('\nAnalyse terminée.')

In [ ]:
import plotly.express as px
import pandas as pd

df_results = pd.DataFrame(results_data)

fig = px.scatter(
    df_results,
    x='time', y='confidence',
    size='n_citations',
    text='question',
    title='Métriques par question (taille = nb citations)',
    template='plotly_dark',
    labels={'time': 'Temps de traitement (s)', 'confidence': 'Score de confiance'},
    color='confidence',
    color_continuous_scale='RdYlGn',
)
fig.update_traces(textposition='top center')
fig.show()